## Deep Learning for Mortality Prediction (DLMP)

### Import packages 

In [1]:
import tensorflow as tf
import numpy as np
import os as os
tfkl = tf.keras.layers
import random

### Import functions

In [2]:
import training_fun_seperate_models as training_functions
import importlib

importlib.reload(training_functions)

<module 'training_fun_seperate_models' from '/Users/paigepark/Desktop/repos/deep-mort/code/uncertainty_models/training_fun_seperate_models.py'>

### Import data

#### State data

In [5]:
state_training = np.loadtxt('../../data/state_training.txt')
state_test = np.loadtxt('../../data/state_test.txt')

#### Country data

In [7]:
country_training = np.loadtxt('../../data/country_training.txt')
country_test = np.loadtxt('../../data/country_test.txt')

In [8]:
country_training.shape

(359594, 5)

#### Combined data

In [9]:
combined_training = np.loadtxt('../../data/combined_training.txt')
combined_test = np.loadtxt('../../data/combined_test.txt')

In [11]:
geos_key = np.load('../../data/geos_key.npy')

In [ ]:
geo_dict = {int(code): geo for geo, code in geos_key}

#### All Country Model

These models are those used in the paper to produce all of main figures/table in the paper.

In [12]:
# prep data
country_train_prepped = training_functions.prep_data(country_training, mode="train", changeratetolog=True)
country_test_prepped = training_functions.prep_data(country_test, mode="test", changeratetolog=True)

In [13]:
# get the proper geography input dimension for model set up 
unique_vals = tf.unique(country_training[:, 0]).y
country_geo_dim = np.array(tf.size(unique_vals)).item()
country_geo_dim = country_geo_dim + 50

In [20]:
# Prepare input features once (shared across all iterations)
training_input_features = (
    tf.convert_to_tensor((country_training[:, 2] - 1959) / 60, dtype=tf.float32),
    tf.convert_to_tensor(country_training[:, 3], dtype=tf.float32),
    tf.convert_to_tensor(country_training[:, 0], dtype=tf.float32),
    tf.convert_to_tensor(country_training[:, 1], dtype=tf.float32),
)

test_input_features = (
    tf.convert_to_tensor((country_test[:, 2] - 1959) / 60, dtype=tf.float32),
    tf.convert_to_tensor(country_test[:, 3], dtype=tf.float32),
    tf.convert_to_tensor(country_test[:, 0], dtype=tf.float32),
    tf.convert_to_tensor(country_test[:, 1], dtype=tf.float32),
)

inputs_train = np.delete(country_training, 4, axis=1)
inputs_test = np.delete(country_test, 4, axis=1)

# Storage for ensemble combination
M = 1
train_mus = []
train_vars = []
test_mus = []
test_vars = []
mean_models = []
error_models = []

In [16]:
# train each ensemble member (mean model + error model) with different seeds
for i in range(1, M + 1):
    # Set reproducible seeds per iteration
    np.random.seed(i)
    tf.random.set_seed(i)
    random.seed(i)
    os.environ['PYTHONHASHSEED'] = str(i)

    print(f"\n{'='*60}")
    print(f"Ensemble member {i}/{M}")
    print(f"{'='*60}")

    mean_model, error_model, mean_val_loss, error_val_loss = training_functions.run_two_stage_model(
        country_train_prepped, country_test_prepped, country_training, country_geo_dim,
        epochs=20, steps_per_epoch=1405, lograte=True
    )

    # Get predictions from the two-stage model
    train_mu = mean_model.predict(training_input_features)
    train_var = error_model.predict(training_input_features)
    train_var = np.maximum(train_var, 1e-6)

    test_mu = mean_model.predict(test_input_features)
    test_var = error_model.predict(test_input_features)
    test_var = np.maximum(test_var, 1e-6)

    # Store for ensemble combination
    train_mus.append(train_mu)
    train_vars.append(train_var)
    test_mus.append(test_mu)
    test_vars.append(test_var)
    mean_models.append(mean_model)
    error_models.append(error_model)

    # Save individual member predictions: [geo, gender, year, age, mu, variance]
    training_predictions = np.column_stack((inputs_train, train_mu, train_var))
    test_predictions = np.column_stack((inputs_test, test_mu, test_var))

    mean_model.save(f"../../models/dl_country_mean_{i}.keras")
    error_model.save(f"../../models/dl_country_error_{i}.keras")
    np.savetxt(f"../../data/dl_country_fitted_{i}.txt", training_predictions)
    np.savetxt(f"../../data/dl_country_forecast_{i}.txt", test_predictions)

    print(f"Ensemble member {i} complete | mean val MSE: {mean_val_loss:.6f} | error val MSE: {error_val_loss:.6f}")


Ensemble member 1/5
=== Stage 1: Training mean model ===
Epoch 1/20
1405/1405 - 6s - 4ms/step - loss: 2.0143 - val_loss: 0.2556 - learning_rate: 1.0000e-03
Epoch 2/20
1405/1405 - 4s - 3ms/step - loss: 0.3295 - val_loss: 0.2748 - learning_rate: 1.0000e-03
Epoch 3/20
1405/1405 - 4s - 3ms/step - loss: 0.2519 - val_loss: 0.1941 - learning_rate: 1.0000e-03
Epoch 4/20
1405/1405 - 4s - 3ms/step - loss: 0.2235 - val_loss: 0.2325 - learning_rate: 1.0000e-03
Epoch 5/20
1405/1405 - 4s - 3ms/step - loss: 0.1998 - val_loss: 0.1994 - learning_rate: 1.0000e-03
Epoch 6/20
1405/1405 - 4s - 3ms/step - loss: 0.1912 - val_loss: 0.2329 - learning_rate: 1.0000e-03
Epoch 7/20
1405/1405 - 4s - 3ms/step - loss: 0.1770 - val_loss: 0.1937 - learning_rate: 2.5000e-04
Epoch 8/20
1405/1405 - 4s - 3ms/step - loss: 0.1787 - val_loss: 0.1933 - learning_rate: 2.5000e-04
Epoch 9/20
1405/1405 - 4s - 3ms/step - loss: 0.1720 - val_loss: 0.1554 - learning_rate: 2.5000e-04
Epoch 10/20
1405/1405 - 4s - 3ms/step - loss: 0.175

In [17]:
# combine ensemble members to get overall predictions and uncertainty decomposition
train_mus = np.array(train_mus)    # (M, N, 1)
train_vars = np.array(train_vars)  # (M, N, 1)
test_mus = np.array(test_mus)      # (M, N, 1)
test_vars = np.array(test_vars)    # (M, N, 1)

# Ensemble mean: average of individual means
ensemble_train_mu = np.mean(train_mus, axis=0)
ensemble_test_mu = np.mean(test_mus, axis=0)

# Ensemble variance: (1/M) * sum(sigma_m^2 + mu_m^2) - mu*^2
ensemble_train_var = np.mean(train_vars + train_mus**2, axis=0) - ensemble_train_mu**2
ensemble_test_var = np.mean(test_vars + test_mus**2, axis=0) - ensemble_test_mu**2

# Decompose into aleatoric (data noise) and epistemic (model disagreement)
aleatoric_train = np.mean(train_vars, axis=0)
epistemic_train = np.mean(train_mus**2, axis=0) - ensemble_train_mu**2

aleatoric_test = np.mean(test_vars, axis=0)
epistemic_test = np.mean(test_mus**2, axis=0) - ensemble_test_mu**2

# 95% prediction intervals
z = 1.96
ensemble_train_std = np.sqrt(ensemble_train_var)
ensemble_test_std = np.sqrt(ensemble_test_var)

train_lower = ensemble_train_mu - z * ensemble_train_std
train_upper = ensemble_train_mu + z * ensemble_train_std
test_lower = ensemble_test_mu - z * ensemble_test_std
test_upper = ensemble_test_mu + z * ensemble_test_std

In [ ]:
# Save: [geo, gender, year, age, mu, total_var, aleatoric_var, epistemic_var, lower_95, upper_95]
ensemble_train_output = np.column_stack((
    inputs_train, ensemble_train_mu, ensemble_train_var,
    aleatoric_train, epistemic_train, train_lower, train_upper
))
ensemble_test_output = np.column_stack((
    inputs_test, ensemble_test_mu, ensemble_test_var,
    aleatoric_test, epistemic_test, test_lower, test_upper
))

np.savetxt("../../data/dl_country_ensemble_fitted.txt", ensemble_train_output)
np.savetxt("../../data/dl_country_ensemble_forecast.txt", ensemble_test_output)

print("\nEnsemble combination complete.")
print(f"Output columns: geo, gender, year, age, mu, total_var, aleatoric_var, epistemic_var, lower_95, upper_95")
print(f"Train shape: {ensemble_train_output.shape}")
print(f"Test shape:  {ensemble_test_output.shape}")



Ensemble combination complete.
Output columns: geo, gender, year, age, mu, total_var, aleatoric_var, epistemic_var, lower_95, upper_95
Train shape: (359594, 10)
Test shape:  (79400, 10)


In [21]:
print(ensemble_train_output[:5])

[[ 5.00000000e+01  0.00000000e+00  1.95900000e+03  0.00000000e+00
  -3.82681918e+00  5.25379181e-03  9.99999997e-07  5.24997711e-03
  -3.96888590e+00 -3.68475246e+00]
 [ 5.00000000e+01  1.00000000e+00  1.95900000e+03  0.00000000e+00
  -3.55219483e+00  1.21412277e-02  9.99999997e-07  1.21393204e-02
  -3.76816177e+00 -3.33622789e+00]
 [ 5.00000000e+01  0.00000000e+00  1.95900000e+03  1.00000000e+00
  -6.37237120e+00  7.32421875e-04  9.99999997e-07  7.32421875e-04
  -6.42541504e+00 -6.31932735e+00]
 [ 5.00000000e+01  1.00000000e+00  1.95900000e+03  1.00000000e+00
  -6.17759562e+00  2.23159790e-03  9.99999997e-07  2.23159790e-03
  -6.27018547e+00 -6.08500576e+00]
 [ 5.00000000e+01  0.00000000e+00  1.95900000e+03  2.00000000e+00
  -6.91837549e+00  7.62939453e-04  9.99999997e-07  7.62939453e-04
  -6.97251320e+00 -6.86423779e+00]]


In [22]:
-3.68475246e+00-(-3.96888590e+00)

0.2841334400000002

In [23]:
-3.33622789e+00-(-3.76816177e+00)

0.4319338799999999